# Colab 환경 구축


### 활용 라이브러리 (고정)

*   [torch==2.7.0](https://pytorch.org/)
*   [pytorch-lightning==2.5.4](https://pypi.org/project/pytorch-lightning/)


### 참고사항

*   GPU 최대 12시간 연속 사용
*   PyTorch의 경우 설치되어있지 않아 매 런타임마다 install command 실행
*   노트북파일 맨 첫 셀에 패키지 설치하는 코드 넣고 사용하는 것을 권장

In [1]:
!pip3 install torch==2.7.0 torchvision torchaudio
!pip3 install pytorch-lightning==2.5.4

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cusparselt-cu12
    Found existing installation: nvidia-cusparselt-cu12 0.7.1
    Uninstalling nvidia-cusparselt-cu12-0.

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pytorch_lightning as pl
import numpy as np
import torch.nn.functional as F
from torchmetrics import functional as FM
import seaborn as sns

In [3]:
def set_seed_everything(seed=42):
    # Fix the random number generator seed
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Use deterministic algorithms and disable benchmark mode
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    pl.seed_everything(seed)

set_seed_everything(42)

INFO:lightning_fabric.utilities.seed:Seed set to 42


## Generate Number Dataset
----------------------------------

### Number Dataset 설명


#### 입력 데이터

Xs : 최소 5자리 최대 20자리 미만의 숫자들의 나열

q  : Xs와 비교할 수 있는 query 숫자 하나

#### 출력 데이터
y : Xs 의 숫자들 중 q보다 큰 최초의 수 혹은 그 사이의 숫자


#### why 사이의 숫자?
--> to verify the blendding effect not a simple pointing effect.

----------------------------------

#### 예제 1)
Xs : 3, 5, 6, 2, 5, 7, 2

q  : 4

Y  : 5   (Xs 에서, 4보다 큰 최초의 숫자는 5가 됨)

#### 예제 2)
Xs : 1, 4, 7, 9, 8

q  : 4

Y  : 6   

#### 위 문제를 풀 수 있으려면, neural network 이 query 의 값보다 큰 최초의 숫자에 대해 attention 할 수 있어야 한다.  


In [4]:
def generate_data(num_examples):

    min_digit = 5
    max_digit = 10

    min_number = 0
    max_number = 10

    data = []
    for i in range(num_examples):
        while True:
            # sequence generation (Xs)
            seq = np.random.randint(min_number, high=max_number, size=10)
            n_digit = np.random.randint(min_digit, high=max_digit, size=1)[0]
            seq = seq[:n_digit]

            # query generation (q)
            min_number_in_seq = np.min(seq)
            max_number_in_seq = np.max(seq)

            if min_number_in_seq == max_number_in_seq: continue
            query = np.random.randint(min_number_in_seq, high=max_number_in_seq, size=1)[0]
            break


        # output generation (y)
        candidates = [ (pos, num) for pos, num in enumerate(seq) if num > query]
        candidates = sorted(candidates, key=lambda x: x[1])
        y_tuple = candidates[0]
        pos_y, num_y = y_tuple

        y_s = list( range(query+1, num_y))
        if len(y_s) == 0: # case --> y = num_y
            y = num_y
        else:
            y = y_s[-1]

        data.append( (list(seq), query, y) )

    return data


def dump_data(data, fn):
    with open(fn, 'w', encoding='utf-8') as f:
        for seq, query, y in data:
            seq_str = ",".join( [str(s) for s in seq] )
            print("{}\t{}\t{}".format(seq_str, query, y), file=f)

        print("# of examples : ", len(data))
        print("Data is dumped at ", fn)




data_root = './data/numbers'

os.makedirs(data_root, exist_ok=True)

fns = {
            'train' : os.path.join(data_root, 'train.txt'),
            'test'  : os.path.join(data_root, 'test.txt')
      }

if not os.path.exists(data_root): os.makedirs(data_root)

all_data = generate_data(num_examples=50000)

train_data = all_data[:45000]
test_data = all_data[45000:]

dump_data(train_data, fns['train'])
dump_data(test_data, fns['test'])


# of examples :  45000
Data is dumped at  ./data/numbers/train.txt
# of examples :  5000
Data is dumped at  ./data/numbers/test.txt


In [5]:
def load_data(fn):
    data = []
    with open(fn, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()

            seq_str, query, y = line.split('\t')
            seqs = seq_str.split(',')
            data.append( (seqs, query, y) )
    return data

In [6]:
class NumberDataset(Dataset):
    """Dataset."""

    def __init__(self, fn, input_vocab, output_vocab, max_seq_length):
        self.input_vocab = input_vocab
        self.output_vocab = output_vocab
        self.max_seq_length = max_seq_length

        # load
        self.data = load_data(fn)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq, q, y = self.data[idx]

        # [ input ]
        seq_ids = [ self.input_vocab[t] for t in seq ]

        # <pad> processing
        pad_id = self.input_vocab['<pad>']
        num_to_fill = self.max_seq_length - len(seq)
        seq_ids = seq_ids + [pad_id]*num_to_fill

        # mask processing (1 for valid, 0 for invalid)
        weights = [1]*len(seq) + [0]*num_to_fill


        # ex)
        # seq_ids : 6, 3, 5, 2, 4, _, _, _
        # weights : 1, 1, 1, 1, 1, 0, 0, 0

        # [ query ]
        # NOTE : we assume that query vocab space is same as input vocab space

        q_id = self.input_vocab[q] # enable valid query
        #q_id = 0 # disable query -- to check query effect in attention mechanism

        # [ ouput ]
        y_id = self.output_vocab[y]

        item = [
                    # input
                    np.array(seq_ids),
                    q_id,
                    np.array(weights),

                    # output
                    y_id
               ]
        return item

#데이터 모듈 정의


In [7]:
class NumberDataModule(pl.LightningDataModule):
    def __init__(self,
                 max_seq_length: int=12,
                 batch_size: int = 32):
        super().__init__()
        self.batch_size = batch_size
        self.max_seq_length = max_seq_length

        input_vocab, output_vocab = self.make_vocab('./data/numbers/train.txt')
        self.input_vocab_size  = len( input_vocab )
        self.output_vocab_size = len( output_vocab )
        self.padding_idx = input_vocab['<pad>']

        self.all_train_dataset = NumberDataset('./data/numbers/train.txt', input_vocab, output_vocab, max_seq_length)
        self.test_dataset      = NumberDataset('./data/numbers/test.txt', input_vocab, output_vocab, max_seq_length)

        self.input_r_vocab  = { v:k for k,v in input_vocab.items() }
        self.output_r_vocab = { v:k for k,v in output_vocab.items() }

        # random split train / valiid for early stopping
        N = len(self.all_train_dataset)
        tr = int(N*0.8) # 8 for the training
        va = N - tr     # 2 for the validation
        self.train_dataset, self.valid_dataset = torch.utils.data.random_split(self.all_train_dataset, [tr, va])


    def make_vocab(self, fn):
        input_tokens = []
        output_tokens = []
        data = load_data(fn)

        for seqs, query, y in data:
            for token in seqs:
                input_tokens.append(token)
            output_tokens.append(y)

        input_tokens = list(set(input_tokens))
        output_tokens = list(set(output_tokens))

        input_tokens.sort()
        output_tokens.sort()

        # [input vocab]
        # add <pad> symbol to input tokens as a first item
        input_tokens = ['<pad>'] + input_tokens
        input_vocab = { str(token):index for index, token in enumerate(input_tokens) }

        # [output voab]
        output_vocab = { str(token):index for index, token in enumerate(output_tokens) }

        return input_vocab, output_vocab

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True) # NOTE : Shuffle

    def val_dataloader(self):
        return DataLoader(self.valid_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)


# BahdanauAttention

In [11]:
class BahdanauAttention(nn.Module):
    """
    Attention > Additive Attention > Bahdanau approach

    Inputs:
        query_vector  : [hidden_size]
        multiple_items: [batch_size, num_of_items, hidden_size]
    Returns:
        blendded_vector:    [batch_size, item_vector hidden_size]
        attention_scores:   [batch_size, num_of_items]
    """
    def __init__(self, item_dim, query_dim, attention_dim):
        super(BahdanauAttention, self).__init__()
        self.item_dim = item_dim            # dim. of multiple item vector
        self.query_dim = query_dim          # dim. of query vector
        self.attention_dim = attention_dim  # dim. of projected item or query vector

        # W is used for project query to the attention dimension
        # U is used for project each item to the attention dimension
        self.W = nn.Linear(self.query_dim,  self.attention_dim, bias=False)
        self.U = nn.Linear(self.item_dim,   self.attention_dim, bias=False)

        # v is used for calculating attention score which is scalar value
        self.v = nn.Parameter(torch.randn(1, attention_dim, dtype=torch.float))

    def _calculate_reactivity(self, query_vector, multiple_items):
        B, N, H = multiple_items.shape  # [B,N,H]

        # linear projection is applied to the last dimension
        query = self.W(query_vector) # [B, H] --> [B, attention_dim]
        items = self.U(multiple_items) # [B, N, H] --> [B, N, attention_dim]

        # note that broadcasting is performed when adding different shape
        reactivity_scores = torch.tanh(query.unsqueeze(1) + items) # [B, 1, attention_dim] + [B, N, attention_dim] --> [B, N, attention_dim]
        reactivity_scores = torch.sum(self.v * reactivity_scores, dim=2) # [B, N, attention_dim] * [attention_dim, 1] --> [B, N]

        return reactivity_scores #[B, N]

    def forward(self, query_vector, multiple_items, mask):
        """
        Inputs:
            query_vector:   [query_vector hidden_size]
            multiple_items: [batch_size, num_of_items, item_vector hidden_size]
            mask:           [batch_size, num_of_items, num_of_items]  1 for valid item, 0 for invalid item
        Returns:
            blendded_vector:    [batch_size, item_vector hidden_size]
            attention_scores:   [batch_size, num_of_items]
        """
        assert mask is not None, "mask is required"

        # B : batch_size, N : number of multiple items, H : hidden size of item
        B, N, H = multiple_items.size()

        # Three Steps
        # 1) [reactivity] try to check the reactivity with ( item_t and query_vector ) N times
        # 2) [masking]    try to penalize invalid items such as <pad>
        # 3) [attention]  try to get proper attention scores (=propability form) over the reactivity scores
        # 4) [blend]      try to blend multiple items with attention scores

        # Step-1) reactivity
        reactivity_scores = self._calculate_reactivity(query_vector, multiple_items)

        # Step-2) masking
        # The mask marks valid positions so we invert it using `mask & 0`.
        # detail : check the masked_fill_() of pytorch
        reactivity_scores.data.masked_fill_(mask == 0, -float('inf'))

        # Step-3) attention score
        attention_scores = F.softmax(reactivity_scores, dim=1) # over the item dimensions

        # Step-4) blend multiple items
        # merge by weighted sum
        attention_scores = attention_scores.unsqueeze(1) # [B, 1, #_of_items]

        # [B, 1, #_of_items] * [B, #_of_items, dim_of_item] --> [B, 1, dim_of_item]
        blendded_vector = torch.matmul(attention_scores, multiple_items)
        blendded_vector = blendded_vector.squeeze(1) # [B, dim_of_item]

        return blendded_vector, attention_scores

In [9]:
class Attention_Number_Finder(pl.LightningModule):
    def __init__(self,
                 # network setting
                 input_vocab_size,
                 output_vocab_size,
                 d_model,      # dim. in attemtion mechanism
                 padding_idx,
                 # optiimzer setting
                 learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()


        # note
        #   - the dimension for query and multi-items do not need to be same.
        #   - for simplicity, we make all the dimensions as same.

        # symbol_number_character to vector_number
        self.digit_emb = nn.Embedding(self.hparams.input_vocab_size,
                                      self.hparams.d_model,
                                      padding_idx=self.hparams.padding_idx)

        # sequence encoder using RNN
        self.encoder = nn.LSTM(d_model, int(self.hparams.d_model/2), # why? since bidirectional LSTM
                            num_layers=2,
                            bidirectional=True,
                            batch_first=True
                          )

        # attention mechanism
        self.att = BahdanauAttention(item_dim=self.hparams.d_model,
                                     query_dim=self.hparams.d_model,
                                     attention_dim=self.hparams.d_model)

        # [to output]
        self.to_output = nn.Linear(self.hparams.d_model, self.hparams.output_vocab_size) # D -> a single number

        # loss
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, seq_ids, q_id, weight):
        # ------------------- ENCODING with ATTENTION -----------------#
        # [ Digit Character Embedding ]
        # seq_ids : [B, max_seq_len]
        seq_embs = self.digit_emb(seq_ids.long()) # [B, max_seq_len, emb_dim]

        # [ Sequence of Numbers Encoding ]
        seq_encs, _ = self.encoder(seq_embs) # [B, max_seq_len, enc_dim*2]  since we have 2 layers

        # dynamic encoding-summarization (blending)
        multiple_items = seq_encs

        # with query (context)
        query = self.digit_emb(q_id) # [B, query_dim]

        blendded_vector, attention_scores = self.att(query, multiple_items, mask=weight) # [B, #_of_items]
        # blendded_vector  : [B, dim_of_sequence_enc]
        # attention_scores : [B, query_len, key_len]

        # To output
        logits = self.to_output(blendded_vector)
        return logits, attention_scores

    def training_step(self, batch, batch_idx):
        seq_ids, q_id, weights, y_id = batch
        logits, _ = self(seq_ids, q_id, weights)  # [B, output_vocab_size]
        loss = self.criterion(logits, y_id.long())
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)

        # all logs are automatically stored for tensorboard
        return loss

    def validation_step(self, batch, batch_idx):
        seq_ids, q_id, weights, y_id = batch

        logits, _ = self(seq_ids, q_id, weights)  # [B, output_vocab_size]
        loss = self.criterion(logits, y_id.long())

        ## get predicted result
        prob = F.softmax(logits, dim=-1)
        acc = FM.accuracy(prob, y_id, task='multiclass',  num_classes=self.hparams.output_vocab_size)
        metrics = {'val_acc': acc, 'val_loss': loss}
        self.log_dict(metrics)
        return metrics

    def validation_step_end(self, val_step_outputs):
        val_acc  = val_step_outputs['val_acc'].cpu()
        val_loss = val_step_outputs['val_loss'].cpu()

        self.log('validation_acc',  val_acc, prog_bar=True)
        self.log('validation_loss', val_loss, prog_bar=True)

    def test_step(self, batch, batch_idx):
        seq_ids, q_id, weights, y_id = batch

        logits, _ = self(seq_ids, q_id, weights)  # [B, output_vocab_size]
        loss = self.criterion(logits, y_id.long())

        ## get predicted result
        prob = F.softmax(logits, dim=-1)
        acc = FM.accuracy(prob, y_id, task='multiclass',  num_classes=self.hparams.output_vocab_size)
        metrics = {'test_acc': acc, 'test_loss': loss}
        self.log_dict(metrics, on_epoch=True)
        return metrics

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer


In [12]:
import sys
from argparse import ArgumentParser
from pytorch_lightning.callbacks import EarlyStopping


# Colab 또는 Jupyter에서 자동으로 전달되는 인자 제거
sys.argv = [sys.argv[0]]

# ------------
# args
# ------------
parser = ArgumentParser()
parser.add_argument('--batch_size', default=200, type=int)
parser.add_argument('--d_model',    default=512, type=int)  # dim. for attention model
parser.add_argument('--learning_rate', default=0.0001, type=float)
args = parser.parse_args()

# ------------
# data
# ------------
dm = NumberDataModule(batch_size=args.batch_size)


# ------------
# model
# ------------
model = Attention_Number_Finder(dm.input_vocab_size,
                                    dm.output_vocab_size,
                                    args.d_model,       # dim. in attemtion mechanism
                                    dm.padding_idx,
                                    args.learning_rate)

# ------------
# training
# ------------
trainer = pl.Trainer(
                            max_epochs=30,
                            callbacks=[EarlyStopping(monitor='val_loss')],
                            accelerator="gpu",
                            devices=1 if torch.cuda.is_available() else None
                    )
trainer.fit(model, datamodule=dm)

# ------------
# testing
# ------------
result = trainer.test(model, dataloaders=dm)
print(result)

# {'test_acc': 0.9477999806404114, 'test_loss': 0.1168440580368042}


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | digit_emb | Embedding         | 5.6 K  | train
1 | encoder   | LSTM              | 3.2 M  | train
2 | att       | BahdanauAttention | 524 K  | train
3 | to_output | Linear            | 4.6 K  | train
4 | criterion | CrossEntropyLoss  | 0      | train
-------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 598, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1011, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1055, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 216, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 455, in advance
    self.epoch_loop.run(self._data_fetcher)
  File "/usr/local/lib/python3.12

TypeError: object of type 'NoneType' has no len()